In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_09_domain_events_mapped
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_09_domain_events_mapped
# Spec Ref  : §1.6 Event Aggregation — Domain-Level (domain_events_mapped)
# Purpose   : Aggregate mapped_events per domain × meta_event × event_day.
#             Incremental MERGE — only reprocesses events since last aggregated day.
#
# Spec DQ gate: domain must be lowercase in all rows
# Run after : map_04 (mapped_events)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 09: domain_events_mapped (incremental MERGE) ===  customer={customer_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_09_domain_events_mapped",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.domain_events_mapped",
    """
        domain       STRING NOT NULL,
        meta_event   STRING NOT NULL,
        event_day    DATE   NOT NULL,
        occurrences  BIGINT,
        tenant       BIGINT
    """,
    cluster_by="domain, meta_event, event_day"
)


In [0]:
# CELL 4 -- Incremental watermark
try:
    last_day = spark.sql(f"""
        SELECT COALESCE(MAX(event_day), DATE('1900-01-01'))
        FROM {sv}.domain_events_mapped
    """).collect()[0][0]
    print(f"  Last aggregated day: {last_day}")

    # CELL 5
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW domain_events_delta AS
        SELECT
            c.domain,
            e.meta_event,
            CAST(e.event_timestamp AS DATE)   AS event_day,
            COUNT(*)                          AS occurrences,
            e.tenant
        FROM {sv}.mapped_events e
        INNER JOIN {sv}.contacts_all c
            ON e.contactid = c.id
        WHERE c.domain IS NOT NULL
          AND CAST(e.event_timestamp AS DATE) >= '{last_day}'
        GROUP BY
            c.domain,
            e.meta_event,
            CAST(e.event_timestamp AS DATE),
            e.tenant
    """)

    # Add tenant column if missing (table may pre-date DDL update)
    existing_cols = [c.name for c in spark.table(f"{sv}.domain_events_mapped").schema]
    if "tenant" not in existing_cols:
        spark.sql(f"ALTER TABLE {sv}.domain_events_mapped ADD COLUMN tenant BIGINT")

    # CELL 6 -- UPDATED
    spark.sql(f"""
        MERGE INTO {sv}.domain_events_mapped AS target
        USING domain_events_delta AS source
        ON  target.domain     = source.domain
        AND target.meta_event = source.meta_event
        AND target.event_day  = source.event_day
        AND COALESCE(target.tenant, -1) = COALESCE(source.tenant, -1)
        WHEN MATCHED THEN
            UPDATE SET
                target.occurrences = source.occurrences,
                target.tenant      = source.tenant
        WHEN NOT MATCHED THEN
            INSERT (domain, meta_event, event_day, occurrences, tenant)
            VALUES (source.domain, source.meta_event, source.event_day,
                    source.occurrences, source.tenant)
    """)

    # CELL 7 -- Spec DQ: domain must be lowercase
    n = cnt(f"{sv}.domain_events_mapped")
    bad = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.domain_events_mapped
        WHERE domain != LOWER(domain)
    """).collect()[0][0]
    print(f"  domain_events_mapped: {n:,} rows")
    status = '✅' if bad == 0 else '❌'
    print(f"  Uppercase domains: {bad}  (spec gate = 0)  {status}")
except Exception as e:
    print(f"❌ domain_events_mapped processing failed: {e}")
    log_audit(
        job_name       = "02b_map_09_domain_events_mapped",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.domain_events_mapped").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_09_domain_events_mapped",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise